# mini-llm: TinyStories baseline on a Kaggle T4

Trains mini-llm (a ~14M-parameter GPT-2-style model built from scratch in this repo) on the
full TinyStories set with one fixed recipe, two seeds, fp16 mixed precision, and records
validation **bits per byte**, throughput and wall time. Everything is driven by
`small-projects/mini-llm/kaggle/run_recipe.py`; this notebook only calls it.

**Before running:** right panel -> *Session options* -> **Accelerator: GPU T4 x2** (only one GPU
is used) and **Internet: On**. For the full run use **Save Version -> Save & Run All (Commit)** so it
keeps running with the browser closed (12 h session limit).

Order: settings -> GPU check -> get code -> **200-step smoke test** (downloads + prepares data
first, ~1.9 GB) -> full recipe (both seeds) -> GGUF export (skipped with a message if this checkout
has no `src/export_gguf.py`) -> `results.md`. Outputs land in `/kaggle/working/minillm-t4/`.
Within a session, re-running a cell is safe: finished work is detected and skipped.

In [ ]:
# Settings
import json
import os

REPO_URL = "https://github.com/angasko-12345/projects-ai.git"
REPO_REF = "main"     # branch, tag or commit. Before the Kaggle PR merges use "upgrade/minillm-kaggle-t4".
RUN_FULL = True       # False: stop after the smoke test (use this for a first interactive check)
ONLY_SEED = None      # 1337 or 4242: one seed per saved version, if the smoke test says both won't fit 12 h
DEVICE = "cuda"
EXTRA_ARGS = []       # e.g. ["--override", "max_steps=6000"]; results are then marked NOT the baseline

# Local test hooks; unset on Kaggle. MINILLM_REPO_DIR reuses an existing checkout.
REPO_DIR = os.environ.get("MINILLM_REPO_DIR")
DEVICE = os.environ.get("MINILLM_DEVICE", DEVICE)
EXTRA_ARGS = EXTRA_ARGS + json.loads(os.environ.get("MINILLM_RECIPE_ARGS", "[]"))
if ONLY_SEED is not None:
    EXTRA_ARGS = EXTRA_ARGS + ["--only-seed", str(ONLY_SEED)]
print("ref:", REPO_REF, "| device:", DEVICE, "| full run:", RUN_FULL, "| extra:", EXTRA_ARGS)

In [ ]:
# GPU check
import torch

print("torch", torch.__version__, "| CUDA build", torch.version.cuda)
if DEVICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError("No GPU in this session. Right panel -> Session options -> "
                           "Accelerator -> GPU T4 x2, then run all cells again.")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    if "T4" not in torch.cuda.get_device_name(0):
        print("note: not a T4; the recipe still runs, but numbers are not comparable to a T4 run")

In [ ]:
# Get the code at REPO_REF and install mini-llm's dependencies
import subprocess
import sys


def sh(cmd, cwd=None):
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"failed: {' '.join(cmd)}")


if REPO_DIR is None:
    REPO_DIR = "/tmp/projects-ai"   # outside /kaggle/working: the code is not part of the saved output
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        sh(["git", "init", "-q", REPO_DIR])
        sh(["git", "remote", "add", "origin", REPO_URL], cwd=REPO_DIR)
    sh(["git", "fetch", "-q", "--depth", "1", "origin", REPO_REF], cwd=REPO_DIR)
    sh(["git", "checkout", "-q", "--force", "FETCH_HEAD"], cwd=REPO_DIR)
sh(["git", "log", "-1", "--format=%H %s"], cwd=REPO_DIR)
PKG = os.path.join(REPO_DIR, "small-projects", "mini-llm")
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=PKG)

sys.path.insert(0, os.path.join(PKG, "kaggle"))
import run_recipe  # noqa: E402


def recipe(stage):
    try:
        run_recipe.main(["--stage", stage, "--device", DEVICE, *EXTRA_ARGS])
    except SystemExit as exc:  # turn a script exit into a cell error, so Run All stops here
        if exc.code not in (0, None):
            raise RuntimeError(str(exc.code)) from None

In [ ]:
# 1) Smoke test: download + prepare TinyStories (first run only), 200 fp16 steps, then a short
#    resume that must restore the GradScaler. Prints tokens/sec and the projected full-run time.
recipe("smoke")

In [ ]:
# 2) Full recipe, every seed (or ONLY_SEED). Within one session, running this cell again skips
#    finished seeds and resumes an unfinished one from its newest checkpoint.
if RUN_FULL:
    recipe("full")
else:
    print("RUN_FULL is False: skipping the full run")

In [ ]:
# 3) GGUF export of the first seed (f16, plus Q8_0 via a pinned llama.cpp build).
#    Skips with a message when this checkout has no src/export_gguf.py.
if RUN_FULL:
    recipe("export")
else:
    print("RUN_FULL is False: nothing to export")

In [ ]:
# 4) results.md from the measured values (blank where something did not run)
recipe("summary")